In [20]:
import pandas as pd

books = pd.read_csv("../data/processed/books_with_categories.csv")

In [21]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = 0,
                      model_kwargs={"use_safetensors": True})
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528676815330982},
  {'label': 'neutral', 'score': 0.005764597095549107},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.002092393347993493},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [22]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [23]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.6548410654067993},
  {'label': 'neutral', 'score': 0.16985194385051727},
  {'label': 'sadness', 'score': 0.11640904098749161},
  {'label': 'surprise', 'score': 0.02070070616900921},
  {'label': 'disgust', 'score': 0.01910071261227131},
  {'label': 'joy', 'score': 0.015161263756453991},
  {'label': 'anger', 'score': 0.003935149405151606}]]

In [24]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(books["description"][0])

sentences = [sent.text for sent in doc.sents]
predictions = classifier(sentences)

In [25]:
sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives.'

In [26]:

predictions[0]

[{'label': 'surprise', 'score': 0.7214335799217224},
 {'label': 'neutral', 'score': 0.1681574136018753},
 {'label': 'fear', 'score': 0.05271957814693451},
 {'label': 'joy', 'score': 0.043484512716531754},
 {'label': 'anger', 'score': 0.009235911071300507},
 {'label': 'disgust', 'score': 0.002850534161552787},
 {'label': 'sadness', 'score': 0.002118478761985898}]

In [27]:
sentences[3]

'Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist.'

In [28]:
predictions[3]

[{'label': 'fear', 'score': 0.9790166020393372},
 {'label': 'neutral', 'score': 0.006491743493825197},
 {'label': 'sadness', 'score': 0.004959614481776953},
 {'label': 'anger', 'score': 0.0033238993491977453},
 {'label': 'surprise', 'score': 0.0027822679840028286},
 {'label': 'disgust', 'score': 0.002765241777524352},
 {'label': 'joy', 'score': 0.0006605989765375853}]

In [29]:
for sent, pred in zip(sentences, predictions):
    print("Sentence:", sent)
    print("Emotion:", pred)

Sentence: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives.
Emotion: [{'label': 'surprise', 'score': 0.7214335799217224}, {'label': 'neutral', 'score': 0.1681574136018753}, {'label': 'fear', 'score': 0.05271957814693451}, {'label': 'joy', 'score': 0.043484512716531754}, {'label': 'anger', 'score': 0.009235911071300507}, {'label': 'disgust', 'score': 0.002850534161552787}, {'label': 'sadness', 'score': 0.002118478761985898}]
Sentence: John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers.
Emotion: [{'label': 'disgust', 'score': 0.45429444313049316}, {'label': 'neutral', 'score': 0.44546249508857727}, {'label': 'joy', 'score': 0.042101435363292694}, {'label': 'sadness', 'score': 0.03025977872312069}, {'label': 'anger', 'score': 0.016755471006035805}, {'label': 'surprise', 'score': 0.007567733060568571}, {'label': 'fear', 'score': 0.0035586075

In [30]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009235911071300507},
 {'label': 'disgust', 'score': 0.002850534161552787},
 {'label': 'fear', 'score': 0.05271957814693451},
 {'label': 'joy', 'score': 0.043484512716531754},
 {'label': 'neutral', 'score': 0.1681574136018753},
 {'label': 'sadness', 'score': 0.002118478761985898},
 {'label': 'surprise', 'score': 0.7214335799217224}]

In [31]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}


In [32]:

for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [33]:
emotion_scores

{'anger': [np.float64(0.06413368880748749),
  np.float64(0.6126189827919006),
  np.float64(0.06413368880748749),
  np.float64(0.3514832854270935),
  np.float64(0.08141238987445831),
  np.float64(0.232224702835083),
  np.float64(0.5381833910942078),
  np.float64(0.06413368880748749),
  np.float64(0.30067017674446106),
  np.float64(0.06413368880748749)],
 'disgust': [np.float64(0.27359139919281006),
  np.float64(0.34828436374664307),
  np.float64(0.10400675237178802),
  np.float64(0.15072230994701385),
  np.float64(0.18449531495571136),
  np.float64(0.7271749377250671),
  np.float64(0.15585504472255707),
  np.float64(0.10400675237178802),
  np.float64(0.27948129177093506),
  np.float64(0.1779269576072693)],
 'fear': [np.float64(0.928167998790741),
  np.float64(0.9425276517868042),
  np.float64(0.9723208546638489),
  np.float64(0.3607073724269867),
  np.float64(0.09504338353872299),
  np.float64(0.051362842321395874),
  np.float64(0.7474281191825867),
  np.float64(0.4044954478740692),
  n

In [34]:

from tqdm import tqdm
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

n = len(books)
isbn = books["isbn13"].tolist()
emotion_scores = {label: np.zeros(n) for label in emotion_labels}

batch_size = 32

for start in tqdm(range(0, n, batch_size)):
    end = min(start + batch_size, n)
    batch_desc = books["description"][start:end].tolist()
    all_sentences = []
    mapping = []

    for idx, desc in enumerate(batch_desc):
        sentences = desc.split(".")
        all_sentences.extend(sentences)
        mapping.extend([idx] * len(sentences))

    predictions = classifier(all_sentences, batch_size=32, truncation=True)

    temp_scores = [{label: 0 for label in emotion_labels} for _ in range(len(batch_desc))]

    for pred, book_idx in zip(predictions, mapping):
        for item in pred:
            label = item["label"].lower()
            score = item["score"]
            if score > temp_scores[book_idx][label]:
                temp_scores[book_idx][label] = score

    for i, scores in enumerate(temp_scores):
        for label in emotion_labels:
            emotion_scores[label][start + i] = scores[label]

100%|██████████| 163/163 [00:54<00:00,  3.00it/s]


In [35]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [36]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273592,0.928168,0.932797,0.967158,0.729603,0.646216,9780002005883
1,0.612619,0.348285,0.942528,0.704421,0.111690,0.252545,0.887940,9780002261982
2,0.064134,0.104007,0.972321,0.767236,0.111690,0.078766,0.549477,9780006178736
3,0.351483,0.150722,0.360708,0.251881,0.111690,0.078766,0.732686,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.475881,0.078766,0.884390,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.148209,0.030642,0.919165,0.255171,0.980877,0.030656,0.853722,9788172235222
5193,0.064134,0.114383,0.051363,0.400263,0.111690,0.227765,0.883198,9788173031014
5194,0.009997,0.009929,0.339217,0.947779,0.066685,0.057625,0.375756,9788179921623
5195,0.064134,0.104007,0.459270,0.759455,0.368111,0.078765,0.951104,9788185300535


In [37]:
books = pd.merge(books, emotions_df, on = "isbn13")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273592,0.928168,0.932797,0.967158,0.729603,0.646216
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612619,0.348285,0.942528,0.704421,0.111690,0.252545,0.887940
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767236,0.111690,0.078766,0.549477
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.150722,0.360708,0.251881,0.111690,0.078766,0.732686
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.475881,0.078766,0.884390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.148209,0.030642,0.919165,0.255171,0.980877,0.030656,0.853722
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.064134,0.114383,0.051363,0.400263,0.111690,0.227765,0.883198
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339217,0.947779,0.066685,0.057625,0.375756
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.064134,0.104007,0.459270,0.759455,0.368111,0.078765,0.951104


In [38]:
books.to_csv("books_with_emotions.csv", index = False)